<a href="https://colab.research.google.com/github/pottigarinikitha123-rgb/Image-Forgery-Detection/blob/main/DeepL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub timm grad-cam

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import timm
from sklearn.metrics import accuracy_score, classification_report
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
import kagglehub

#Download latest version of the dataset
path = kagglehub.dataset_download("fari4117/cdffake-v2-dataset")
print("Raw kagglehub path:", path)

#Actual images are inside CDFFAKE V2 Dataset
data_dir = os.path.join(path, "CDFFAKE V2 Dataset")
print("Image root:", data_dir)

In [ ]:
#Data preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

#ImageFolder expects class subfolders (Deepfake, Real, Tempered)
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
print("Classes:", dataset.classes)

num_classes = len(dataset.classes)

#Train/test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_ds, test_ds = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

print("Train samples:", len(train_ds))
print("Test samples:", len(test_ds))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
cnn_model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=num_classes
)

cnn_model = cnn_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_cnn = optim.Adam(cnn_model.parameters(), lr=1e-4)

In [ ]:
EPOCHS_CNN = 5

for epoch in range(EPOCHS_CNN):
    cnn_model.train()
    total_loss = 0.0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer_cnn.zero_grad()
        outputs = cnn_model(images)              # [B, 3]
        loss = criterion(outputs, labels)        # labels in {0,1,2}
        loss.backward()
        optimizer_cnn.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"[CNN] Epoch {epoch+1}/{EPOCHS_CNN} | Loss: {avg_loss:.4f}")

torch.save(cnn_model.state_dict(), "cnn_forgery_model.pth")
print("CNN model saved to cnn_forgery_model.pth")

In [ ]:
cnn_model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = cnn_model(images)
        preds = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

acc = accuracy_score(y_true, y_pred)
print("CNN Test Accuracy:", acc)
print("\nCNN Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=dataset.classes))

In [ ]:
vit_model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=num_classes
)

vit_model = vit_model.to(device)
optimizer_vit = optim.Adam(vit_model.parameters(), lr=2e-5)

In [ ]:
EPOCHS_VIT = 3

for epoch in range(EPOCHS_VIT):
    vit_model.train()
    total_loss = 0.0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer_vit.zero_grad()
        outputs = vit_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_vit.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"[ViT] Epoch {epoch+1}/{EPOCHS_VIT} | Loss: {avg_loss:.4f}")

torch.save(vit_model.state_dict(), "vit_forgery_model.pth")
print("ViT model saved to vit_forgery_model.pth")

In [ ]:
vit_model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = vit_model(images)
        preds = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

acc = accuracy_score(y_true, y_pred)
print("ViT Test Accuracy:", acc)
print("\nViT Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=dataset.classes))

In [ ]:
#Pick the last EfficientNet block as target layer
target_layers = [cnn_model.blocks[-1]]

cam = GradCAM(
    model=cnn_model,
    target_layers=target_layers
)

#Take one example from test set
img_tensor, label_idx = test_ds[0]
input_tensor = img_tensor.unsqueeze(0).to(device)

cnn_model.eval()
grayscale_cam = cam(input_tensor=input_tensor)[0]

In [ ]:
#Prepare original image for overlay
rgb_img = np.transpose(img_tensor.numpy(), (1, 2, 0))
rgb_img = (rgb_img * 0.5 + 0.5).clip(0, 1)

cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

plt.figure(figsize=(4,4))
plt.imshow(cam_image)
plt.title(f"Forgery Detection Heatmap (True: {dataset.classes[label_idx]})")
plt.axis("off")
plt.show()

In [ ]:
class_names = dataset.classes

def predict_image(path, model, device=device):
    model.eval()
    img = Image.open(path).convert("RGB")
    img_t = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_t)
        pred_idx = outputs.argmax(dim=1).item()

    return class_names[pred_idx]

#Example in use
print(predict_image("/content/post10272025.png", cnn_model))